# 🏀 March Machine Learning Mania 2026 — Rank #1 Strategy Notebook
## Advanced Ensemble: RandomForest + XGBoost + CatBoost + LightGBM + PyTorch Neural Network

> **Goal:** Predict the probability that Team1 beats Team2 in every possible NCAA tournament matchup.  
> **Metric:** Log Loss (lower is better — target: 0.00000 via perfect calibration on tourney data).  
> **Strategy:** Elo ratings + seed differentials + advanced box-score features + Massey ordinals + deep ensemble with stacking + calibrated + clipped predictions.

---
### 📋 Notebook Outline
1. Import Libraries & GPU Configuration
2. Load & Explore All Datasets  
3. Data Preprocessing & Cleaning
4. Feature Engineering — Team Season Stats
5. Feature Engineering — Elo Ratings, Seeds, Win Ratios
6. Feature Engineering — Recent Form, Massey Ordinals, Conference Strength
7. Build Training & Test Datasets
8. Model 1: Random Forest with Hyperparameter Tuning
9. Model 2: XGBoost + Optuna (GPU)
10. Model 3: CatBoost + Optuna (GPU)
11. Model 4: LightGBM + Optuna (GPU)
12. Model 5: PyTorch Neural Network (GPU)
13. Ensemble & Meta-Learner Stacking
14. Calibration & Prediction Clipping
15. Generate Final Submission

In [ ]:
# ============================================================
# ▶ RUN THIS CELL FIRST — installs any missing packages
# ============================================================
import subprocess, sys

def _pip(pkg):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg, '-q'])

for pkg, install_name in [
    ('xgboost',   'xgboost>=2.0'),
    ('lightgbm',  'lightgbm>=4.0'),
    ('catboost',  'catboost>=1.2'),
    ('optuna',    'optuna>=3.0'),
    ('torch',     'torch'),
    ('tqdm',      'tqdm'),
]:
    try:
        __import__(pkg)
        print(f"  ✅ {pkg}")
    except ImportError:
        print(f"  ⚙️  installing {install_name}...")
        _pip(install_name)
        print(f"  ✅ {pkg} installed")

print("\n✅ All packages ready!")
print("💡 Kaggle GPU: Settings → Accelerator → GPU T4 x2")

---
## 🗺️ Quick Start Guide

| Step | Action |
|------|--------|
| **Local** | Run cells top-to-bottom. Data path auto-set to your local folder. |
| **Kaggle** | Enable **GPU T4 x2** in *Settings → Accelerator*. Path auto-set to `/kaggle/input/march-machine-learning-mania-2026`. |
| **Output** | `submission.csv` (full ensemble) + `submission_elo_baseline.csv` (Elo-only) |

### 🏆 Key Strategies Used (Rank #1 Research)
1. **Elo Ratings** — K=20 with Margin-of-Victory multiplier (better than flat Elo)
2. **Advanced Box Score** — Offensive/Defensive Rating, eFG%, Possessions, NetRtg
3. **Massey Ordinals** — Aggregate of 100+ computer ranking systems pre-tournament
4. **Recent Form** — Last-10-games momentum beats season averages
5. **Symmetric Training** — Double every game row (Team1/Team2 swap) removes ordering bias
6. **Optuna Hypertuning** — 100 trials per model with 5-fold CV
7. **GPU Acceleration** — XGBoost `gpu_hist`, LightGBM `device=gpu`, CatBoost `GPU`, PyTorch CUDA
8. **Meta-Stacking** — Logistic regression on OOF predictions + scipy weight optimization
9. **Isotonic Calibration** — Ensures probabilities are well-calibrated for log-loss
10. **Prediction Clipping** — `[0.025, 0.975]` avoids catastrophic log-loss penalties

> **Note:** The LB score of 0.00000 is achieved when your predictions perfectly match the actual tournament outcomes. Since we're predicting *before* the tournament, the goal is the **best possible calibrated probability**, not a perfect oracle.

## ⚙️ Section 1: Import Required Libraries & Configuration

In [ ]:
# ============================================================
# SECTION 1 — IMPORTS & CONFIGURATION
# ============================================================
import os, sys, warnings, logging, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from tqdm.auto import tqdm

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 100)
pd.set_option('display.max_rows', 50)
logging.basicConfig(level=logging.INFO)

# ── Reproducibility ──────────────────────────────────────────
SEED = 42
np.random.seed(SEED)

# ── Paths — Auto-detect Kaggle vs Local ──────────────────────
LOCAL_DATA_PATH   = '/Users/badalkr.sharma/Documents/Kaggle Competitions/March Machine Learning Mania 2026/data'
LOCAL_OUTPUT_PATH = '/Users/badalkr.sharma/Documents/Kaggle Competitions/March Machine Learning Mania 2026'

# Kaggle stores competition data at /kaggle/input/competitions/<slug>
KAGGLE_DATA_PATH   = '/kaggle/input/competitions/march-machine-learning-mania-2026'
KAGGLE_OUTPUT_PATH = '/kaggle/working'

# Priority: Kaggle competitions folder → Kaggle input folder → Local
if os.path.exists(KAGGLE_DATA_PATH):
    DATA_PATH   = KAGGLE_DATA_PATH
    OUTPUT_PATH = KAGGLE_OUTPUT_PATH
    print("🟢 Running on KAGGLE (competitions path)")
elif os.path.exists('/kaggle/input/march-machine-learning-mania-2026'):
    DATA_PATH   = '/kaggle/input/march-machine-learning-mania-2026'
    OUTPUT_PATH = KAGGLE_OUTPUT_PATH
    print("🟢 Running on KAGGLE (input path)")
elif os.path.exists(LOCAL_DATA_PATH):
    DATA_PATH   = LOCAL_DATA_PATH
    OUTPUT_PATH = LOCAL_OUTPUT_PATH
    print("🔵 Running LOCALLY")
else:
    # Fallback: search for any folder containing MTeams.csv
    for root, dirs, files in os.walk('/kaggle'):
        if 'MTeams.csv' in files:
            DATA_PATH   = root
            OUTPUT_PATH = KAGGLE_OUTPUT_PATH
            print(f"🟡 Found data at: {DATA_PATH}")
            break
    else:
        DATA_PATH   = LOCAL_DATA_PATH
        OUTPUT_PATH = LOCAL_OUTPUT_PATH
        print("⚠️  Could not auto-detect path — using local default")

print(f"\n📂 DATA_PATH   : {DATA_PATH}")
print(f"📂 OUTPUT_PATH : {OUTPUT_PATH}")
print(f"📂 Files found : {len(os.listdir(DATA_PATH)) if os.path.exists(DATA_PATH) else '❌ PATH NOT FOUND'}")

# ── ML / DL Libraries ────────────────────────────────────────
from sklearn.model_selection import StratifiedKFold, KFold, cross_val_score
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import log_loss, roc_auc_score
from sklearn.calibration import CalibratedClassifierCV, calibration_curve
from sklearn.impute import SimpleImputer
from scipy import optimize
import xgboost as xgb
import lightgbm as lgb
from catboost import CatBoostClassifier, Pool
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

# ── PyTorch ──────────────────────────────────────────────────
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

# ── GPU Detection ─────────────────────────────────────────────
if torch.cuda.is_available():
    DEVICE = torch.device('cuda')
    GPU_AVAILABLE = True
    print(f"\n🚀 GPU Detected : {torch.cuda.get_device_name(0)}")
    print(f"   Memory       : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
elif torch.backends.mps.is_available():
    DEVICE = torch.device('mps')   # Apple Silicon
    GPU_AVAILABLE = True
    print("\n🚀 Apple Silicon MPS GPU detected")
else:
    DEVICE = torch.device('cpu')
    GPU_AVAILABLE = False
    print("\n⚠️  No GPU found — running on CPU (slower training)")

print(f"\n✅ PyTorch  : {torch.__version__}")
print(f"✅ XGBoost  : {xgb.__version__}")
print(f"✅ LightGBM : {lgb.__version__}")
print(f"✅ Device   : {DEVICE}")

## 📊 Section 2: Load & Explore All Datasets

In [ ]:
# ============================================================
# SECTION 2 — LOAD ALL DATASETS
# ============================================================

# ── Diagnostic: list all files in DATA_PATH ──────────────────
if os.path.exists(DATA_PATH):
    available_files = set(os.listdir(DATA_PATH))
    print(f"📁 Files available in DATA_PATH ({len(available_files)} total):")
    for f in sorted(available_files):
        print(f"   {f}")
else:
    available_files = set()
    print(f"❌ DATA_PATH does not exist: {DATA_PATH}")

def load_csv(fname):
    """Load CSV — tries exact name, then case-insensitive match."""
    # Direct match
    path = os.path.join(DATA_PATH, fname)
    if os.path.exists(path):
        df = pd.read_csv(path)
        print(f"  ✅ {fname:<52} shape={df.shape}")
        return df
    # Case-insensitive fallback (handles Kaggle path variations)
    fname_lower = fname.lower()
    for f in available_files:
        if f.lower() == fname_lower:
            path = os.path.join(DATA_PATH, f)
            df = pd.read_csv(path)
            print(f"  ✅ {f:<52} shape={df.shape}  (matched as '{fname}')")
            return df
    print(f"  ⚠️  NOT FOUND: {fname}")
    return None

print("\n" + "=" * 60)
print("LOADING MEN'S DATA")
print("=" * 60)
MTeams               = load_csv('MTeams.csv')
MSeasons             = load_csv('MSeasons.csv')
MTeamConferences     = load_csv('MTeamConferences.csv')
MTeamCoaches         = load_csv('MTeamCoaches.csv')
MConferenceTourneyGames = load_csv('MConferenceTourneyGames.csv')
MGameCities          = load_csv('MGameCities.csv')
MRegSeason           = load_csv('MRegularSeasonCompactResults.csv')
MRegSeasonDet        = load_csv('MRegularSeasonDetailedResults.csv')
MTourneyCompact      = load_csv('MNCAATourneyCompactResults.csv')
MTourneyDetailed     = load_csv('MNCAATourneyDetailedResults.csv')
MTourneySeeds        = load_csv('MNCAATourneySeeds.csv')
MTourneySlots        = load_csv('MNCAATourneySlots.csv')
MTourneySeedRounds   = load_csv('MNCAATourneySeedRoundSlots.csv')
MSecondaryTourney    = load_csv('MSecondaryTourneyCompactResults.csv')
MSecondaryTeams      = load_csv('MSecondaryTourneyTeams.csv')
MMasseyOrdinals      = load_csv('MMasseyOrdinals.csv')

print("\n" + "=" * 60)
print("LOADING WOMEN'S DATA")
print("=" * 60)
WTeams               = load_csv('WTeams.csv')
WSeasons             = load_csv('WSeasons.csv')
WTeamConferences     = load_csv('WTeamConferences.csv')
WRegSeason           = load_csv('WRegularSeasonCompactResults.csv')
WRegSeasonDet        = load_csv('WRegularSeasonDetailedResults.csv')
WTourneyCompact      = load_csv('WNCAATourneyCompactResults.csv')
WTourneyDetailed     = load_csv('WNCAATourneyDetailedResults.csv')
WTourneySeeds        = load_csv('WNCAATourneySeeds.csv')

print("\n" + "=" * 60)
print("LOADING SUBMISSION FILES")
print("=" * 60)
SubStage1 = load_csv('SampleSubmissionStage1.csv')
SubStage2 = load_csv('SampleSubmissionStage2.csv')

# ── Validate critical files loaded ────────────────────────────
critical = {
    'MRegSeasonDet'  : MRegSeasonDet,
    'MTourneyCompact': MTourneyCompact,
    'MTourneySeeds'  : MTourneySeeds,
    'WRegSeasonDet'  : WRegSeasonDet,
    'WTourneyCompact': WTourneyCompact,
    'WTourneySeeds'  : WTourneySeeds,
}
missing = [k for k, v in critical.items() if v is None]
if missing:
    raise RuntimeError(f"❌ Critical files missing: {missing}\n"
                       f"   Check DATA_PATH = {DATA_PATH}")

# Pick whichever submission file is available
if SubStage2 is not None:
    sub_raw = SubStage2.copy()
    print(f"\n✅ Using Stage 2 submission: {len(sub_raw):,} rows")
elif SubStage1 is not None:
    sub_raw = SubStage1.copy()
    print(f"\n✅ Using Stage 1 submission: {len(sub_raw):,} rows")
else:
    raise RuntimeError("❌ No submission file found!")

print(f"\n📋 Predictions to generate: {len(sub_raw):,}")
print("\nSample Submission:")
display(sub_raw.head())

## 🧹 Section 3: Data Preprocessing & Cleaning

In [ ]:
# ============================================================
# SECTION 3 — DATA PREPROCESSING & CLEANING
# ============================================================

# ── Add Gender Tag & Combine Detailed Results ─────────────────
MRegSeasonDet['Gender'] = 'M'
WRegSeasonDet['Gender'] = 'W'
MTourneyDetailed['Gender'] = 'M'
WTourneyDetailed['Gender'] = 'W'

# Combine regular season detailed (primary feature source, from 2003)
AllRegSeason = pd.concat([MRegSeasonDet, WRegSeasonDet], ignore_index=True)
AllTourney   = pd.concat([MTourneyDetailed, WTourneyDetailed], ignore_index=True)
AllRegSeason = AllRegSeason[AllRegSeason['Season'] >= 2003].reset_index(drop=True)
AllTourney   = AllTourney[AllTourney['Season'] >= 2003].reset_index(drop=True)

# Compact tourney (all years — needed for Elo going back to 1985)
MTourneyCompact['Gender'] = 'M'
WTourneyCompact['Gender'] = 'W'
AllTourneyCompact = pd.concat([MTourneyCompact, WTourneyCompact], ignore_index=True)

# Regular season compact (all years — for Elo initialization)
MRegSeason['Gender'] = 'M'
WRegSeason['Gender'] = 'W'
AllRegSeasonCompact = pd.concat([MRegSeason, WRegSeason], ignore_index=True)

# ── Seed Parsing ──────────────────────────────────────────────
MTourneySeeds['Gender'] = 'M'
WTourneySeeds['Gender'] = 'W'
AllSeeds = pd.concat([MTourneySeeds, WTourneySeeds], ignore_index=True)

def parse_seed(seed_str):
    """Convert seed string 'W01a' → 1,  'Z16' → 16."""
    import re
    num = re.findall(r'\d+', str(seed_str))
    return int(num[0]) if num else np.nan

AllSeeds['SeedNum']    = AllSeeds['Seed'].apply(parse_seed)
AllSeeds['SeedRegion'] = AllSeeds['Seed'].str[0]

print("✅ AllRegSeason shape    :", AllRegSeason.shape)
print("✅ AllTourney shape      :", AllTourney.shape)
print("✅ AllTourneyCompact     :", AllTourneyCompact.shape)
print("✅ AllRegSeasonCompact   :", AllRegSeasonCompact.shape)
print("✅ AllSeeds shape        :", AllSeeds.shape)
print(f"\nSeasons available (Men tourney): "
      f"{MTourneyCompact['Season'].min()} – {MTourneyCompact['Season'].max()}")
print(f"Seasons available (detailed)   : "
      f"{AllRegSeason['Season'].min()} – {AllRegSeason['Season'].max()}")

## 🔢 Section 4: Feature Engineering — Team Season Statistics

In [ ]:
# ============================================================
# SECTION 4 — TEAM SEASON STATISTICS (Advanced Box Score)
# ============================================================

def compute_team_season_stats(df_detailed):
    """
    From detailed regular season results, compute per-team, per-season
    advanced statistics. Each game appears twice (W and L side).
    """
    # ── Winner side ──────────────────────────────────────────
    w = df_detailed[['Season','Gender','WTeamID',
                      'WScore','LScore','WLoc','NumOT',
                      'WFGM','WFGA','WFGM3','WFGA3','WFTM','WFTA',
                      'WOR','WDR','WAst','WTO','WStl','WBlk','WPF',
                      'LFGM','LFGA','LOR','LDR','LAst','LTO']].copy()
    w.columns = ['Season','Gender','TeamID',
                 'PtsFor','PtsAgainst','Loc','NumOT',
                 'FGM','FGA','FGM3','FGA3','FTM','FTA',
                 'ORB','DRB','Ast','TO','Stl','Blk','PF',
                 'OppFGM','OppFGA','OppORB','OppDRB','OppAst','OppTO']
    w['Win'] = 1

    # ── Loser side ───────────────────────────────────────────
    l = df_detailed[['Season','Gender','LTeamID',
                      'LScore','WScore','WLoc','NumOT',
                      'LFGM','LFGA','LFGM3','LFGA3','LFTM','LFTA',
                      'LOR','LDR','LAst','LTO','LStl','LBlk','LPF',
                      'WFGM','WFGA','WOR','WDR','WAst','WTO']].copy()
    l.columns = ['Season','Gender','TeamID',
                 'PtsFor','PtsAgainst','OppLoc','NumOT',
                 'FGM','FGA','FGM3','FGA3','FTM','FTA',
                 'ORB','DRB','Ast','TO','Stl','Blk','PF',
                 'OppFGM','OppFGA','OppORB','OppDRB','OppAst','OppTO']
    l['Win'] = 0
    # Flip location perspective
    loc_map = {'H': 'A', 'A': 'H', 'N': 'N'}
    l['Loc'] = l['OppLoc'].map(loc_map).fillna('N')
    l.drop(columns=['OppLoc'], inplace=True)

    games = pd.concat([w, l], ignore_index=True)

    # ── Possession & Efficiency ────────────────────────────────
    # Possessions ≈ FGA - ORB + TO + 0.475*FTA  (Kubatko formula)
    games['Poss']  = games['FGA'] - games['ORB'] + games['TO'] + 0.475 * games['FTA']
    games['OppPoss'] = games['OppFGA'] - games['OppORB'] + games['OppTO'] + 0.475 * games['FTA']
    games['OrtgRaw'] = 100 * games['PtsFor']    / games['Poss'].replace(0, np.nan)
    games['DrtgRaw'] = 100 * games['PtsAgainst']/ games['OppPoss'].replace(0, np.nan)
    games['NetRtg']  = games['OrtgRaw'] - games['DrtgRaw']
    games['FG_pct']  = games['FGM']  / games['FGA'].replace(0, np.nan)
    games['FG3_pct'] = games['FGM3'] / games['FGA3'].replace(0, np.nan)
    games['FT_pct']  = games['FTM']  / games['FTA'].replace(0, np.nan)
    games['TRB']     = games['ORB']  + games['DRB']
    games['PointDiff']= games['PtsFor'] - games['PtsAgainst']

    # ── Aggregate per Season × Team ──────────────────────────
    agg = games.groupby(['Season','Gender','TeamID']).agg(
        G            = ('Win','count'),
        Wins         = ('Win','sum'),
        PtsFor       = ('PtsFor','mean'),
        PtsAgainst   = ('PtsAgainst','mean'),
        PointDiff    = ('PointDiff','mean'),
        FG_pct       = ('FG_pct','mean'),
        FG3_pct      = ('FG3_pct','mean'),
        FT_pct       = ('FT_pct','mean'),
        FGM          = ('FGM','mean'),
        FGA          = ('FGA','mean'),
        FGM3         = ('FGM3','mean'),
        FGA3         = ('FGA3','mean'),
        FTM          = ('FTM','mean'),
        FTA          = ('FTA','mean'),
        ORB          = ('ORB','mean'),
        DRB          = ('DRB','mean'),
        TRB          = ('TRB','mean'),
        Ast          = ('Ast','mean'),
        TO           = ('TO','mean'),
        Stl          = ('Stl','mean'),
        Blk          = ('Blk','mean'),
        PF           = ('PF','mean'),
        Poss         = ('Poss','mean'),
        OrtgRaw      = ('OrtgRaw','mean'),
        DrtgRaw      = ('DrtgRaw','mean'),
        NetRtg       = ('NetRtg','mean'),
        OppFGM       = ('OppFGM','mean'),
        OppFGA       = ('OppFGA','mean'),
    ).reset_index()

    agg['WinRate']       = agg['Wins'] / agg['G']
    agg['OppFG_pct']     = agg['OppFGM'] / agg['OppFGA'].replace(0, np.nan)
    agg['AstTO_ratio']   = agg['Ast'] / agg['TO'].replace(0, np.nan)
    agg['StlBlk']        = agg['Stl'] + agg['Blk']
    agg['ThreeRate']     = agg['FGA3'] / agg['FGA'].replace(0, np.nan)
    agg['FTRate']        = agg['FTA']  / agg['FGA'].replace(0, np.nan)
    agg['ORB_pct']       = agg['ORB']  / agg['TRB'].replace(0, np.nan)
    agg['DRB_pct']       = agg['DRB']  / agg['TRB'].replace(0, np.nan)

    return agg

print("⚙️  Computing team season statistics...")
TeamStats = compute_team_season_stats(AllRegSeason)
print(f"✅ TeamStats shape: {TeamStats.shape}")
print(f"   Seasons: {TeamStats['Season'].min()} – {TeamStats['Season'].max()}")
print(f"   Unique teams: {TeamStats['TeamID'].nunique()}")
display(TeamStats.head(3))

## 🏆 Section 5: Feature Engineering — Elo Ratings, Seeds & Win Ratios

In [ ]:
# ============================================================
# SECTION 5 — ELO RATINGS (Per Gender, All Seasons)
# ============================================================

def compute_elo(games_df, K=20, init_elo=1500, hfa=100, mov_factor=True):
    """
    Compute season-end Elo ratings using all regular season + tourney games.
    games_df columns: Season, WTeamID, LTeamID, WScore, LScore, Gender
    Returns: dict  {(Season, TeamID, Gender): elo}
    """
    elo = {}  # {(TeamID, Gender): current_elo}

    def get_elo(tid, gender):
        return elo.get((tid, gender), init_elo)

    # Sort chronologically
    games = games_df.sort_values(['Season','DayNum']).reset_index(drop=True)

    # Track end-of-season snapshots
    season_elo = {}   # {(Season, TeamID, Gender): elo_at_end_of_season}
    current_season = {}  # stores last season processed per (team, gender)

    for _, row in games.iterrows():
        s = row['Season']
        wid, lid = row['WTeamID'], row['LTeamID']
        g = row['Gender']
        loc = row.get('WLoc', 'N')

        ew = get_elo(wid, g)
        el = get_elo(lid, g)

        # Home field adjustment
        if loc == 'H':
            ew_adj = ew + hfa
        elif loc == 'A':
            ew_adj = ew - hfa
        else:
            ew_adj = ew

        # Expected win probability
        expected_w = 1 / (1 + 10 ** ((el - ew_adj) / 400))

        # Margin of victory multiplier (FiveThirtyEight style)
        if mov_factor and 'WScore' in row and 'LScore' in row:
            score_diff = abs(row['WScore'] - row['LScore'])
            mov = np.log(max(score_diff, 1) + 1) * (2.2 / ((ew - el) * 0.001 + 2.2))
            k_adj = K * mov
        else:
            k_adj = K

        new_ew = ew + k_adj * (1 - expected_w)
        new_el = el + k_adj * (0 - (1 - expected_w))

        elo[(wid, g)] = new_ew
        elo[(lid, g)] = new_el

        # Also store most recent elo value indexed by season
        season_elo[(s, wid, g)] = new_ew
        season_elo[(s, lid, g)] = new_el

    # Convert to DataFrame
    records = [{'Season': s, 'TeamID': t, 'Gender': gdr, 'Elo': v}
               for (s, t, gdr), v in season_elo.items()]
    elo_df = pd.DataFrame(records)
    return elo_df

print("⚙️  Computing Elo ratings (all games, both genders)...")

# Use COMPACT results to get broader season coverage (incl. pre-2003)
AllGames = pd.concat([
    AllRegSeasonCompact[['Season','Gender','WTeamID','LTeamID','WScore','LScore','WLoc','DayNum']],
    AllTourneyCompact[['Season','Gender','WTeamID','LTeamID','WScore','LScore','WLoc','DayNum']],
], ignore_index=True)

EloDF = compute_elo(AllGames, K=20, init_elo=1500, hfa=100, mov_factor=True)

print(f"✅ Elo ratings computed: {len(EloDF):,} entries")
print(f"   Seasons covered: {EloDF['Season'].min()} – {EloDF['Season'].max()}")
print(EloDF[EloDF['Gender']=='M'].nlargest(10,'Elo')[['Season','TeamID','Elo']])

In [ ]:
# ============================================================
# SECTION 5b — SEED FEATURES & HISTORICAL TOURNEY WIN RATES
# ============================================================

# ── Historical tourney win rate per team (all-time) ─────────
def compute_tourney_history(tourney_compact):
    """Compute all-time tourney win/loss per (team, gender) → win rate."""
    w = tourney_compact[['Season','Gender','WTeamID']].copy()
    w.columns = ['Season','Gender','TeamID']
    w['TourneyWin'] = 1

    l = tourney_compact[['Season','Gender','LTeamID']].copy()
    l.columns = ['Season','Gender','TeamID']
    l['TourneyWin'] = 0

    games = pd.concat([w, l], ignore_index=True)
    # Cumulative up to (but not including) each season
    games_sorted = games.sort_values('Season')

    # All-time tourney win rate (up to and including prior seasons)
    records = []
    for gender in ['M','W']:
        g_df = games_sorted[games_sorted['Gender']==gender]
        for season in sorted(g_df['Season'].unique()):
            hist = g_df[g_df['Season'] < season]
            if len(hist) == 0:
                continue
            agg = hist.groupby('TeamID').agg(
                TourneyGames = ('TourneyWin','count'),
                TourneyWins  = ('TourneyWin','sum'),
            ).reset_index()
            agg['TourneyWinRate'] = agg['TourneyWins'] / agg['TourneyGames']
            agg['Season'] = season
            agg['Gender'] = gender
            records.append(agg)
    return pd.concat(records, ignore_index=True)

print("⚙️  Computing historical tourney win rates...")
TourneyHistory = compute_tourney_history(AllTourneyCompact)
print(f"✅ TourneyHistory: {TourneyHistory.shape}")

# ── Historical head-to-head (H2H) win rate ───────────────────
def compute_h2h(tourney_compact):
    """Compute all-time H2H win rate for every (T1, T2, Gender) pair."""
    df = tourney_compact[['Season','Gender','WTeamID','LTeamID']].copy()
    df['key'] = df.apply(lambda r: (min(r['WTeamID'],r['LTeamID']),
                                    max(r['WTeamID'],r['LTeamID'])), axis=1)
    df['T1'] = df['key'].apply(lambda x: x[0])
    df['T2'] = df['key'].apply(lambda x: x[1])
    df['T1Win'] = (df['WTeamID'] == df['T1']).astype(int)
    agg = df.groupby(['Gender','T1','T2']).agg(
        H2H_Games = ('T1Win','count'),
        H2H_T1Wins= ('T1Win','sum'),
    ).reset_index()
    agg['H2H_WinRate'] = agg['H2H_T1Wins'] / agg['H2H_Games']
    return agg

print("⚙️  Computing head-to-head tourney records...")
H2H = compute_h2h(AllTourneyCompact)
print(f"✅ H2H records: {H2H.shape}")

print("\n✅ Seed parsing from AllSeeds:")
print(AllSeeds[['Season','Gender','TeamID','SeedNum']].head())

## 📈 Section 6: Feature Engineering — Recent Form, Massey Ordinals & Conference Strength

In [ ]:
# ============================================================
# SECTION 6 — RECENT FORM, MASSEY ORDINALS & CONFERENCE STRENGTH
# ============================================================

# ── Last-N Games Form ─────────────────────────────────────────
def compute_recent_form(compact_df, n_games=10):
    """
    For each team/season, compute last-n regular-season win rate
    and avg point differential (momentum indicator).
    """
    w = compact_df[['Season','Gender','WTeamID','LTeamID','WScore','LScore','DayNum']].copy()
    w.rename(columns={'WTeamID':'TeamID','LTeamID':'OppID','WScore':'Pts','LScore':'OppPts'}, inplace=True)
    w['Win'] = 1

    l = compact_df[['Season','Gender','LTeamID','WTeamID','LScore','WScore','DayNum']].copy()
    l.rename(columns={'LTeamID':'TeamID','WTeamID':'OppID','LScore':'Pts','WScore':'OppPts'}, inplace=True)
    l['Win'] = 0

    all_games = pd.concat([w, l], ignore_index=True)
    all_games['Diff'] = all_games['Pts'] - all_games['OppPts']
    all_games = all_games.sort_values(['Season','Gender','TeamID','DayNum'])

    form_records = []
    for (s, g, t), grp in all_games.groupby(['Season','Gender','TeamID']):
        last_n = grp.tail(n_games)
        form_records.append({
            'Season': s, 'Gender': g, 'TeamID': t,
            f'WinRate_Last{n_games}': last_n['Win'].mean(),
            f'AvgDiff_Last{n_games}': last_n['Diff'].mean(),
            f'AvgPts_Last{n_games}': last_n['Pts'].mean(),
        })
    return pd.DataFrame(form_records)

print("⚙️  Computing recent form (last 10 games)...")
RecentForm = compute_recent_form(AllRegSeasonCompact, n_games=10)
print(f"✅ RecentForm: {RecentForm.shape}")

# ── Massey Ordinals (composite ranking) ──────────────────────
print("\n⚙️  Processing Massey Ordinals...")
if MMasseyOrdinals is not None:
    # Use only rankings close to Selection Sunday (day 110+)
    MasseyLate = MMasseyOrdinals[MMasseyOrdinals['RankingDayNum'] >= 110].copy()
    MasseyAgg = MasseyLate.groupby(['Season','TeamID']).agg(
        AvgRank   = ('OrdinalRank','mean'),
        MinRank   = ('OrdinalRank','min'),
        MaxRank   = ('OrdinalRank','max'),
        StdRank   = ('OrdinalRank','std'),
        NumSystems= ('SystemName','nunique'),
    ).reset_index()
    MasseyAgg['Gender'] = 'M'   # Massey ordinals are Men's only
    print(f"✅ MasseyAgg: {MasseyAgg.shape}  "
          f"(seasons {MasseyAgg['Season'].min()}–{MasseyAgg['Season'].max()})")
else:
    MasseyAgg = pd.DataFrame(columns=['Season','TeamID','Gender',
                                       'AvgRank','MinRank','MaxRank','StdRank','NumSystems'])
    print("⚠️  MMasseyOrdinals not available — skipping Massey features")

# ── Conference Strength (avg Elo per conference per season) ──
print("\n⚙️  Computing conference strength...")

# Build AllConferences safely
conf_pieces = []
if MTeamConferences is not None:
    mc = MTeamConferences.copy();  mc['Gender'] = 'M';  conf_pieces.append(mc)
if WTeamConferences is not None:
    wc = WTeamConferences.copy();  wc['Gender'] = 'W';  conf_pieces.append(wc)

if conf_pieces:
    AllConferences = pd.concat(conf_pieces, ignore_index=True)
    ConfStrength = AllConferences.merge(
        EloDF, on=['Season','TeamID','Gender'], how='left'
    ).groupby(['Season','Gender','ConfAbbrev']).agg(
        ConfAvgElo  = ('Elo','mean'),
        ConfMaxElo  = ('Elo','max'),
        ConfMinElo  = ('Elo','min'),
        ConfNumTeams= ('TeamID','nunique'),
    ).reset_index()
    TeamConf = AllConferences.merge(ConfStrength, on=['Season','Gender','ConfAbbrev'], how='left')
    print(f"✅ TeamConf (with conf strength): {TeamConf.shape}")
else:
    # Fallback: empty DataFrame so merge steps don't crash
    TeamConf = pd.DataFrame(columns=['Season','TeamID','Gender','ConfAbbrev',
                                      'ConfAvgElo','ConfMaxElo','ConfMinElo','ConfNumTeams'])
    print("⚠️  Conference files not available — skipping conference features")

## 🏗️ Section 7: Build Training & Test Datasets

In [ ]:
# ============================================================
# SECTION 7a — MASTER FEATURE MERGE FUNCTION
# ============================================================

def merge_all_features(matchup_df, gender='M'):
    """
    Given a DataFrame with columns [Season, Team1, Team2, Gender],
    merge all engineered features for both teams and compute differentials.
    """
    ts = TeamStats[TeamStats['Gender'] == gender].copy()
    el = EloDF[EloDF['Gender'] == gender].copy()
    sd = AllSeeds[AllSeeds['Gender'] == gender][['Season','TeamID','SeedNum']].copy()
    rf = RecentForm[RecentForm['Gender'] == gender].copy()
    th = TourneyHistory[TourneyHistory['Gender'] == gender].copy()
    tc = TeamConf[TeamConf['Gender'] == gender][['Season','TeamID','ConfAbbrev',
                                                   'ConfAvgElo','ConfMaxElo','ConfNumTeams']].copy()
    if gender == 'M':
        ms = MasseyAgg.copy()
    else:
        ms = pd.DataFrame(columns=['Season','TeamID','AvgRank','MinRank','MaxRank','StdRank'])

    df = matchup_df.copy()

    # ── Merge Team Stats ──────────────────────────────────────
    stat_cols = [c for c in ts.columns if c not in ['Season','Gender','TeamID']]
    for side, team_col in [('T1','Team1'), ('T2','Team2')]:
        merged = df.merge(ts.rename(columns={'TeamID': team_col}),
                          on=['Season', team_col], how='left')
        for c in stat_cols:
            df[f'{side}_{c}'] = merged[c].values

    # ── Merge Elo ─────────────────────────────────────────────
    for side, team_col in [('T1','Team1'), ('T2','Team2')]:
        merged = df.merge(el.rename(columns={'TeamID': team_col}),
                          on=['Season', team_col], how='left')
        df[f'{side}_Elo'] = merged['Elo'].values

    # ── Merge Seeds ───────────────────────────────────────────
    for side, team_col in [('T1','Team1'), ('T2','Team2')]:
        merged = df.merge(sd.rename(columns={'TeamID': team_col}),
                          on=['Season', team_col], how='left')
        df[f'{side}_Seed'] = merged['SeedNum'].values

    # ── Merge Recent Form ────────────────────────────────────
    rf_cols = [c for c in rf.columns if c not in ['Season','Gender','TeamID']]
    for side, team_col in [('T1','Team1'), ('T2','Team2')]:
        merged = df.merge(rf.rename(columns={'TeamID': team_col}),
                          on=['Season', team_col], how='left')
        for c in rf_cols:
            df[f'{side}_{c}'] = merged[c].values

    # ── Merge Tourney History ────────────────────────────────
    th_cols = [c for c in th.columns if c not in ['Season','Gender','TeamID']]
    for side, team_col in [('T1','Team1'), ('T2','Team2')]:
        merged = df.merge(th[th['Gender']==gender].rename(columns={'TeamID': team_col}),
                          on=['Season', team_col], how='left')
        for c in th_cols:
            df[f'{side}_{c}'] = merged[c].values

    # ── Merge Conference Strength ────────────────────────────
    tc_feat_cols = ['ConfAvgElo','ConfMaxElo','ConfNumTeams']
    for side, team_col in [('T1','Team1'), ('T2','Team2')]:
        merged = df.merge(tc.rename(columns={'TeamID': team_col}),
                          on=['Season', team_col], how='left')
        for c in tc_feat_cols:
            df[f'{side}_{c}'] = merged[c].values

    # ── Merge Massey Ordinals (Men only) ─────────────────────
    if gender == 'M' and len(ms) > 0:
        ms_cols = [c for c in ms.columns if c not in ['Season','Gender','TeamID']]
        for side, team_col in [('T1','Team1'), ('T2','Team2')]:
            merged = df.merge(ms.rename(columns={'TeamID': team_col}),
                              on=['Season', team_col], how='left')
            for c in ms_cols:
                df[f'{side}_Massey_{c}'] = merged[c].values

    # ── H2H Win Rate ─────────────────────────────────────────
    h2h_m = H2H[H2H['Gender']==gender].copy()
    # Align T1 <= T2 ordering
    df['_T1_lo'] = df[['Team1','Team2']].min(axis=1)
    df['_T2_hi'] = df[['Team1','Team2']].max(axis=1)
    df['_T1_is_lower'] = (df['Team1'] == df['_T1_lo']).astype(int)
    merged_h2h = df.merge(
        h2h_m.rename(columns={'T1':'_T1_lo','T2':'_T2_hi'}),
        on=['_T1_lo','_T2_hi'], how='left'
    )
    raw_h2h = merged_h2h['H2H_WinRate'].fillna(0.5).values
    # Flip if T1 is the "higher" team
    df['H2H_WinRate_T1'] = np.where(df['_T1_is_lower'] == 1, raw_h2h, 1 - raw_h2h)
    df.drop(columns=['_T1_lo','_T2_hi','_T1_is_lower'], inplace=True)

    # ── Differential Features ────────────────────────────────
    diff_base = ['WinRate','PtsFor','PtsAgainst','PointDiff',
                 'FG_pct','FG3_pct','FT_pct','ORB','DRB','TRB',
                 'Ast','TO','Stl','Blk','Poss','OrtgRaw','DrtgRaw',
                 'NetRtg','AstTO_ratio','StlBlk','ThreeRate','FTRate',
                 'ORB_pct','DRB_pct','OppFG_pct',
                 'Elo','Seed',
                 'WinRate_Last10','AvgDiff_Last10','AvgPts_Last10',
                 'TourneyWins','TourneyWinRate','TourneyGames',
                 'ConfAvgElo','ConfMaxElo']
    if gender == 'M':
        diff_base += ['Massey_AvgRank','Massey_MinRank','Massey_MaxRank']

    for b in diff_base:
        c1 = f'T1_{b}'
        c2 = f'T2_{b}'
        if c1 in df.columns and c2 in df.columns:
            df[f'Diff_{b}'] = df[c1] - df[c2]

    # Special ratio features
    if 'T1_Elo' in df.columns and 'T2_Elo' in df.columns:
        df['EloRatio'] = df['T1_Elo'] / (df['T2_Elo'] + 1e-6)
        df['EloWinProb'] = 1 / (1 + 10**((df['T2_Elo'] - df['T1_Elo'])/400))
    if 'T1_Seed' in df.columns and 'T2_Seed' in df.columns:
        df['SeedSum'] = df['T1_Seed'] + df['T2_Seed']
        df['SeedProduct'] = df['T1_Seed'] * df['T2_Seed']
        df['SeedRatio'] = df['T1_Seed'] / (df['T2_Seed'] + 1e-6)

    return df

print("✅ merge_all_features() defined")

In [ ]:
# ============================================================
# SECTION 7b — BUILD TRAINING DATASET
# ============================================================

def build_training_data(tourney_detailed, tourney_compact, gender='M'):
    """
    Build symmetric training pairs from tourney results.
    Each actual game generates TWO rows (Team1=W, Team2=L) AND (Team1=L, Team2=W)
    to remove ordering bias. Label=1 means Team1 wins.
    """
    # Use detailed where available, fill with compact
    det_cols = ['Season','Gender','WTeamID','LTeamID']
    det = tourney_detailed[det_cols + [c for c in tourney_detailed.columns
                                        if c not in det_cols]].copy()
    # Compact for seasons not in detailed
    det_seasons = set(det['Season'].unique())
    comp_extra  = tourney_compact[~tourney_compact['Season'].isin(det_seasons)].copy()

    for df_ in [det, comp_extra]:
        df_['Gender'] = gender

    all_games = pd.concat([
        det[['Season','Gender','WTeamID','LTeamID']],
        comp_extra[['Season','Gender','WTeamID','LTeamID']]
    ], ignore_index=True)
    all_games = all_games[all_games['Gender'] == gender].copy()

    rows = []
    for _, row in all_games.iterrows():
        s = row['Season']
        w, l = row['WTeamID'], row['LTeamID']
        # Row 1: T1=Winner
        rows.append({'Season': s, 'Gender': gender, 'Team1': w, 'Team2': l, 'Target': 1})
        # Row 2: T1=Loser (symmetric)
        rows.append({'Season': s, 'Gender': gender, 'Team1': l, 'Team2': w, 'Target': 0})

    return pd.DataFrame(rows)

print("⚙️  Building training data (Men's)...")
train_pairs_M = build_training_data(MTourneyDetailed, MTourneyCompact, gender='M')
print(f"  Men's tourney pairs: {len(train_pairs_M):,} rows, {train_pairs_M['Season'].nunique()} seasons")

print("⚙️  Building training data (Women's)...")
train_pairs_W = build_training_data(WTourneyDetailed, WTourneyCompact, gender='W')
print(f"  Women's tourney pairs: {len(train_pairs_W):,} rows, {train_pairs_W['Season'].nunique()} seasons")

# ── Merge all features ────────────────────────────────────────
print("\n⚙️  Merging features for Men's training data...")
train_M = merge_all_features(train_pairs_M, gender='M')
print(f"  Men's train with features: {train_M.shape}")

print("⚙️  Merging features for Women's training data...")
train_W = merge_all_features(train_pairs_W, gender='W')
print(f"  Women's train with features: {train_W.shape}")

# Add gender indicator
train_M['IsWomens'] = 0
train_W['IsWomens'] = 1

# ── Combine both genders ──────────────────────────────────────
TrainFull = pd.concat([train_M, train_W], ignore_index=True)
print(f"\n✅ Combined training data: {TrainFull.shape}")
print(f"   Target distribution: {TrainFull['Target'].value_counts().to_dict()}")

In [ ]:
# ============================================================
# SECTION 7c — BUILD TEST DATASET FROM SAMPLE SUBMISSION
# ============================================================

def parse_submission(sub_df):
    """Parse 'Season_T1_T2' ID into (Season, Team1, Team2)."""
    split = sub_df['ID'].str.split('_', expand=True)
    df = sub_df.copy()
    df['Season'] = split[0].astype(int)
    df['Team1']  = split[1].astype(int)
    df['Team2']  = split[2].astype(int)
    return df

# sub_raw was already set in Section 2 (Stage2 preferred, else Stage1)
print(f"📋 Submission file: {len(sub_raw):,} rows")

TestDF = parse_submission(sub_raw)
season_test = sorted(TestDF['Season'].unique())
print(f"   Test seasons: {season_test}")

# Determine gender: Men's team IDs are < 3000, Women's are >= 3000
TestDF['Gender'] = TestDF['Team1'].apply(lambda x: 'W' if x >= 3000 else 'M')
print(f"   Gender split : {TestDF['Gender'].value_counts().to_dict()}")

# ── Merge features for test data ─────────────────────────────
test_M = TestDF[TestDF['Gender']=='M'].copy()
test_W = TestDF[TestDF['Gender']=='W'].copy()

print("\n⚙️  Merging features for Men's test data...")
test_M_feat = merge_all_features(test_M, gender='M')
test_M_feat['IsWomens'] = 0

print("⚙️  Merging features for Women's test data...")
test_W_feat = merge_all_features(test_W, gender='W')
test_W_feat['IsWomens'] = 1

TestFull = pd.concat([test_M_feat, test_W_feat], ignore_index=True)
print(f"\n✅ Test data with features: {TestFull.shape}")

# ── Feature Columns ───────────────────────────────────────────
NON_FEATURE_COLS = {'Season','Gender','Team1','Team2','Target','ID',
                    'Pred','WTeamID','LTeamID','T1_Gender','T2_Gender',
                    'T1_ConfAbbrev','T2_ConfAbbrev','H2H_Gender',
                    'T1_SeedRegion','T2_SeedRegion'}

FEAT_COLS = [
    c for c in TrainFull.columns
    if c not in NON_FEATURE_COLS
    and c != 'Target'
    and TrainFull[c].dtype in [np.float64, np.float32, np.int64, np.int32, float, int]
    and c in TestFull.columns
]
print(f"\n✅ Feature columns : {len(FEAT_COLS)}")
print(f"   First 10       : {FEAT_COLS[:10]}")

# ── Impute NaN ────────────────────────────────────────────────
imputer = SimpleImputer(strategy='median')
X_train_raw = imputer.fit_transform(TrainFull[FEAT_COLS].values)
y_train      = TrainFull['Target'].values
X_test_raw   = imputer.transform(TestFull[FEAT_COLS].values)

print(f"\n✅ X_train : {X_train_raw.shape}  NaN={np.isnan(X_train_raw).sum()}")
print(f"✅ X_test  : {X_test_raw.shape}   NaN={np.isnan(X_test_raw).sum()}")
print(f"✅ y_train : {y_train.shape}  balance={y_train.mean():.3f}")

In [ ]:
# ============================================================
# SECTION 7d — TRAIN/VALIDATION SPLIT & CROSS-VALIDATION SETUP
# ============================================================

# Validation = last 2 seasons of historical tourney data
val_seasons = sorted(TrainFull['Season'].unique())[-2:]
train_seasons = [s for s in TrainFull['Season'].unique() if s not in val_seasons]

print(f"Training seasons  : {min(train_seasons)} – {max(train_seasons)} ({len(train_seasons)} seasons)")
print(f"Validation seasons: {val_seasons}")

train_mask = TrainFull['Season'].isin(train_seasons)
val_mask   = TrainFull['Season'].isin(val_seasons)

X_tr   = X_train_raw[train_mask]
y_tr   = y_train[train_mask]
X_val  = X_train_raw[val_mask]
y_val  = y_train[val_mask]

print(f"\n✅ Train: {X_tr.shape}, Val: {X_val.shape}")
print(f"   Val log-loss of constant 0.5: {log_loss(y_val, np.full(len(y_val), 0.5)):.5f}")

# K-Fold for OOF predictions
N_FOLDS = 5
kf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)

# Storage for OOF and test predictions
oof_preds = {}
test_preds = {}

print(f"\n✅ Using {N_FOLDS}-fold StratifiedKFold")

## 🌲 Section 8: Model 1 — Random Forest with Hyperparameter Tuning

In [ ]:
# ============================================================
# SECTION 8 — RANDOM FOREST WITH OPTUNA TUNING
# ============================================================
import time

def rf_objective(trial):
    params = {
        'n_estimators'     : trial.suggest_int('n_estimators', 200, 800),
        'max_depth'        : trial.suggest_int('max_depth', 5, 20),
        'min_samples_split': trial.suggest_int('min_samples_split', 2, 20),
        'min_samples_leaf' : trial.suggest_int('min_samples_leaf', 1, 10),
        'max_features'     : trial.suggest_categorical('max_features', ['sqrt','log2',0.5,0.7]),
        'n_jobs'           : -1,
        'random_state'     : SEED,
    }
    scores = []
    for tr_idx, vl_idx in kf.split(X_train_raw, y_train):
        Xtr, Xvl = X_train_raw[tr_idx], X_train_raw[vl_idx]
        ytr, yvl = y_train[tr_idx], y_train[vl_idx]
        clf = RandomForestClassifier(**params)
        clf.fit(Xtr, ytr)
        pred = clf.predict_proba(Xvl)[:, 1]
        scores.append(log_loss(yvl, pred))
    return np.mean(scores)

print("⚙️  Tuning Random Forest with Optuna (30 trials)...")
t0 = time.time()
study_rf = optuna.create_study(direction='minimize', sampler=optuna.samplers.TPESampler(seed=SEED))
study_rf.optimize(rf_objective, n_trials=30, show_progress_bar=True)
print(f"   Best log-loss: {study_rf.best_value:.5f}  (took {time.time()-t0:.0f}s)")
print(f"   Best params  : {study_rf.best_params}")

# ── Train final RF on full data with OOF ─────────────────────
best_rf_params = study_rf.best_params
best_rf_params.update({'n_jobs': -1, 'random_state': SEED})

oof_rf   = np.zeros(len(X_train_raw))
test_rf  = np.zeros(len(X_test_raw))

print("\n⚙️  Training Random Forest OOF...")
for fold, (tr_idx, vl_idx) in enumerate(kf.split(X_train_raw, y_train)):
    Xtr, Xvl = X_train_raw[tr_idx], X_train_raw[vl_idx]
    ytr, yvl = y_train[tr_idx], y_train[vl_idx]
    m = RandomForestClassifier(**best_rf_params)
    m.fit(Xtr, ytr)
    oof_rf[vl_idx] = m.predict_proba(Xvl)[:,1]
    test_rf += m.predict_proba(X_test_raw)[:,1] / N_FOLDS
    print(f"  Fold {fold+1} val log-loss: {log_loss(yvl, oof_rf[vl_idx]):.5f}")

oof_preds['RF']  = oof_rf
test_preds['RF'] = test_rf
print(f"\n✅ RF OOF log-loss: {log_loss(y_train, oof_rf):.5f}")

# ── Feature Importance Plot ───────────────────────────────────
final_rf = RandomForestClassifier(**best_rf_params)
final_rf.fit(X_train_raw, y_train)
fi = pd.Series(final_rf.feature_importances_, index=FEAT_COLS).nlargest(25)
plt.figure(figsize=(10, 7))
fi.sort_values().plot(kind='barh', color='steelblue')
plt.title('Top 25 Random Forest Feature Importances', fontsize=14)
plt.tight_layout()
plt.show()

## ⚡ Section 9: Model 2 — XGBoost with Optuna (GPU Accelerated)

In [ ]:
# ============================================================
# SECTION 9 — XGBOOST WITH OPTUNA + GPU
# ============================================================

# GPU tree method selection
XGB_TREE_METHOD = 'gpu_hist' if GPU_AVAILABLE else 'hist'
XGB_DEVICE      = 'cuda'     if GPU_AVAILABLE else 'cpu'
print(f"XGBoost tree method: {XGB_TREE_METHOD}")

def xgb_objective(trial):
    params = {
        'n_estimators'      : trial.suggest_int('n_estimators', 300, 2000),
        'max_depth'         : trial.suggest_int('max_depth', 3, 10),
        'learning_rate'     : trial.suggest_float('learning_rate', 0.005, 0.3, log=True),
        'subsample'         : trial.suggest_float('subsample', 0.5, 1.0),
        'colsample_bytree'  : trial.suggest_float('colsample_bytree', 0.4, 1.0),
        'min_child_weight'  : trial.suggest_int('min_child_weight', 1, 10),
        'reg_alpha'         : trial.suggest_float('reg_alpha', 1e-8, 10.0, log=True),
        'reg_lambda'        : trial.suggest_float('reg_lambda', 1e-8, 10.0, log=True),
        'gamma'             : trial.suggest_float('gamma', 0, 5.0),
        'tree_method'       : XGB_TREE_METHOD,
        'device'            : XGB_DEVICE,
        'eval_metric'       : 'logloss',
        'use_label_encoder' : False,
        'random_state'      : SEED,
        'objective'         : 'binary:logistic',
    }
    scores = []
    for tr_idx, vl_idx in kf.split(X_train_raw, y_train):
        Xtr, Xvl = X_train_raw[tr_idx], X_train_raw[vl_idx]
        ytr, yvl = y_train[tr_idx], y_train[vl_idx]
        m = xgb.XGBClassifier(**params, early_stopping_rounds=50, verbosity=0)
        m.fit(Xtr, ytr, eval_set=[(Xvl, yvl)], verbose=False)
        pred = m.predict_proba(Xvl)[:,1]
        scores.append(log_loss(yvl, pred))
    return np.mean(scores)

print("⚙️  Tuning XGBoost with Optuna (100 trials)...")
t0 = time.time()
study_xgb = optuna.create_study(direction='minimize',
                                  sampler=optuna.samplers.TPESampler(seed=SEED))
study_xgb.optimize(xgb_objective, n_trials=100, show_progress_bar=True)
print(f"   Best log-loss: {study_xgb.best_value:.5f}  (took {time.time()-t0:.0f}s)")
print(f"   Best params  : {study_xgb.best_params}")

# ── OOF Training ─────────────────────────────────────────────
best_xgb_p = study_xgb.best_params.copy()
best_xgb_p.update({
    'tree_method': XGB_TREE_METHOD, 'device': XGB_DEVICE,
    'eval_metric': 'logloss', 'use_label_encoder': False,
    'random_state': SEED, 'objective': 'binary:logistic',
})

oof_xgb  = np.zeros(len(X_train_raw))
test_xgb = np.zeros(len(X_test_raw))

print("\n⚙️  Training XGBoost OOF...")
for fold, (tr_idx, vl_idx) in enumerate(kf.split(X_train_raw, y_train)):
    Xtr, Xvl = X_train_raw[tr_idx], X_train_raw[vl_idx]
    ytr, yvl = y_train[tr_idx], y_train[vl_idx]
    m = xgb.XGBClassifier(**best_xgb_p, early_stopping_rounds=100, verbosity=0)
    m.fit(Xtr, ytr, eval_set=[(Xvl, yvl)], verbose=False)
    oof_xgb[vl_idx] = m.predict_proba(Xvl)[:,1]
    test_xgb += m.predict_proba(X_test_raw)[:,1] / N_FOLDS
    print(f"  Fold {fold+1} val log-loss: {log_loss(yvl, oof_xgb[vl_idx]):.5f}")

oof_preds['XGB']  = oof_xgb
test_preds['XGB'] = test_xgb
print(f"\n✅ XGBoost OOF log-loss: {log_loss(y_train, oof_xgb):.5f}")

# Feature importance
fi_xgb = pd.Series(final_rf.feature_importances_, index=FEAT_COLS)
xgb_fi = pd.Series(
    xgb.XGBClassifier(**best_xgb_p).fit(X_train_raw, y_train).feature_importances_,
    index=FEAT_COLS
).nlargest(20)
plt.figure(figsize=(10, 7))
xgb_fi.sort_values().plot(kind='barh', color='darkorange')
plt.title('Top 20 XGBoost Feature Importances', fontsize=14)
plt.tight_layout()
plt.show()

## 🐱 Section 10: Model 3 — CatBoost with Optuna (GPU Accelerated)

In [ ]:
# ============================================================
# SECTION 10 — CATBOOST WITH OPTUNA + GPU
# ============================================================

CB_TASK_TYPE = 'GPU' if GPU_AVAILABLE else 'CPU'
print(f"CatBoost task type: {CB_TASK_TYPE}")

def catboost_objective(trial):
    params = {
        'iterations'        : trial.suggest_int('iterations', 300, 2000),
        'depth'             : trial.suggest_int('depth', 4, 10),
        'learning_rate'     : trial.suggest_float('learning_rate', 0.005, 0.3, log=True),
        'l2_leaf_reg'       : trial.suggest_float('l2_leaf_reg', 1e-3, 20.0, log=True),
        'bagging_temperature': trial.suggest_float('bagging_temperature', 0.0, 1.0),
        'random_strength'   : trial.suggest_float('random_strength', 0.0, 2.0),
        'border_count'      : trial.suggest_int('border_count', 32, 255),
        'task_type'         : CB_TASK_TYPE,
        'eval_metric'       : 'Logloss',
        'loss_function'     : 'Logloss',
        'random_seed'       : SEED,
        'verbose'           : 0,
        'od_type'           : 'Iter',
        'od_wait'           : 50,
    }
    scores = []
    for tr_idx, vl_idx in kf.split(X_train_raw, y_train):
        Xtr, Xvl = X_train_raw[tr_idx], X_train_raw[vl_idx]
        ytr, yvl = y_train[tr_idx], y_train[vl_idx]
        train_pool = Pool(Xtr, ytr)
        val_pool   = Pool(Xvl, yvl)
        m = CatBoostClassifier(**params)
        m.fit(train_pool, eval_set=val_pool, use_best_model=True)
        pred = m.predict_proba(Xvl)[:,1]
        scores.append(log_loss(yvl, pred))
    return np.mean(scores)

print("⚙️  Tuning CatBoost with Optuna (50 trials)...")
t0 = time.time()
study_cb = optuna.create_study(direction='minimize',
                                sampler=optuna.samplers.TPESampler(seed=SEED))
study_cb.optimize(catboost_objective, n_trials=50, show_progress_bar=True)
print(f"   Best log-loss: {study_cb.best_value:.5f}  (took {time.time()-t0:.0f}s)")
print(f"   Best params  : {study_cb.best_params}")

# ── OOF Training ─────────────────────────────────────────────
best_cb_p = study_cb.best_params.copy()
best_cb_p.update({
    'task_type': CB_TASK_TYPE, 'eval_metric': 'Logloss',
    'loss_function': 'Logloss', 'random_seed': SEED, 'verbose': 0,
    'od_type': 'Iter', 'od_wait': 100,
})

oof_cb   = np.zeros(len(X_train_raw))
test_cb  = np.zeros(len(X_test_raw))

print("\n⚙️  Training CatBoost OOF...")
for fold, (tr_idx, vl_idx) in enumerate(kf.split(X_train_raw, y_train)):
    Xtr, Xvl = X_train_raw[tr_idx], X_train_raw[vl_idx]
    ytr, yvl = y_train[tr_idx], y_train[vl_idx]
    m = CatBoostClassifier(**best_cb_p)
    m.fit(Pool(Xtr, ytr), eval_set=Pool(Xvl, yvl), use_best_model=True)
    oof_cb[vl_idx] = m.predict_proba(Xvl)[:,1]
    test_cb += m.predict_proba(X_test_raw)[:,1] / N_FOLDS
    print(f"  Fold {fold+1} val log-loss: {log_loss(yvl, oof_cb[vl_idx]):.5f}")

oof_preds['CB']  = oof_cb
test_preds['CB'] = test_cb
print(f"\n✅ CatBoost OOF log-loss: {log_loss(y_train, oof_cb):.5f}")

## 💡 Section 11: Model 4 — LightGBM with Optuna (GPU Accelerated)

In [ ]:
# ============================================================
# SECTION 11 — LIGHTGBM WITH OPTUNA + GPU
# ============================================================

LGB_DEVICE = 'gpu' if GPU_AVAILABLE else 'cpu'
print(f"LightGBM device: {LGB_DEVICE}")

def lgb_objective(trial):
    params = {
        'num_leaves'        : trial.suggest_int('num_leaves', 20, 300),
        'max_depth'         : trial.suggest_int('max_depth', 3, 12),
        'learning_rate'     : trial.suggest_float('learning_rate', 0.005, 0.3, log=True),
        'n_estimators'      : trial.suggest_int('n_estimators', 300, 2000),
        'subsample'         : trial.suggest_float('subsample', 0.5, 1.0),
        'colsample_bytree'  : trial.suggest_float('colsample_bytree', 0.4, 1.0),
        'reg_alpha'         : trial.suggest_float('reg_alpha', 1e-8, 10.0, log=True),
        'reg_lambda'        : trial.suggest_float('reg_lambda', 1e-8, 10.0, log=True),
        'min_child_samples' : trial.suggest_int('min_child_samples', 5, 100),
        'min_split_gain'    : trial.suggest_float('min_split_gain', 0.0, 2.0),
        'device'            : LGB_DEVICE,
        'metric'            : 'binary_logloss',
        'objective'         : 'binary',
        'verbose'           : -1,
        'random_state'      : SEED,
    }
    scores = []
    callbacks = [lgb.early_stopping(100, verbose=False), lgb.log_evaluation(-1)]
    for tr_idx, vl_idx in kf.split(X_train_raw, y_train):
        Xtr, Xvl = X_train_raw[tr_idx], X_train_raw[vl_idx]
        ytr, yvl = y_train[tr_idx], y_train[vl_idx]
        m = lgb.LGBMClassifier(**params)
        m.fit(Xtr, ytr,
              eval_set=[(Xvl, yvl)],
              callbacks=callbacks)
        pred = m.predict_proba(Xvl)[:,1]
        scores.append(log_loss(yvl, pred))
    return np.mean(scores)

print("⚙️  Tuning LightGBM with Optuna (100 trials)...")
t0 = time.time()
study_lgb = optuna.create_study(direction='minimize',
                                  sampler=optuna.samplers.TPESampler(seed=SEED))
study_lgb.optimize(lgb_objective, n_trials=100, show_progress_bar=True)
print(f"   Best log-loss: {study_lgb.best_value:.5f}  (took {time.time()-t0:.0f}s)")
print(f"   Best params  : {study_lgb.best_params}")

# ── OOF Training ─────────────────────────────────────────────
best_lgb_p = study_lgb.best_params.copy()
best_lgb_p.update({
    'device': LGB_DEVICE, 'metric': 'binary_logloss',
    'objective': 'binary', 'verbose': -1, 'random_state': SEED,
})
callbacks = [lgb.early_stopping(150, verbose=False), lgb.log_evaluation(-1)]

oof_lgb  = np.zeros(len(X_train_raw))
test_lgb = np.zeros(len(X_test_raw))

print("\n⚙️  Training LightGBM OOF...")
for fold, (tr_idx, vl_idx) in enumerate(kf.split(X_train_raw, y_train)):
    Xtr, Xvl = X_train_raw[tr_idx], X_train_raw[vl_idx]
    ytr, yvl = y_train[tr_idx], y_train[vl_idx]
    m = lgb.LGBMClassifier(**best_lgb_p)
    m.fit(Xtr, ytr, eval_set=[(Xvl, yvl)], callbacks=callbacks)
    oof_lgb[vl_idx] = m.predict_proba(Xvl)[:,1]
    test_lgb += m.predict_proba(X_test_raw)[:,1] / N_FOLDS
    print(f"  Fold {fold+1} val log-loss: {log_loss(yvl, oof_lgb[vl_idx]):.5f}")

oof_preds['LGB']  = oof_lgb
test_preds['LGB'] = test_lgb
print(f"\n✅ LightGBM OOF log-loss: {log_loss(y_train, oof_lgb):.5f}")

# Feature importance
lgb_fi = pd.Series(
    lgb.LGBMClassifier(**best_lgb_p).fit(X_train_raw, y_train).feature_importances_,
    index=FEAT_COLS
).nlargest(20)
plt.figure(figsize=(10,7))
lgb_fi.sort_values().plot(kind='barh', color='seagreen')
plt.title('Top 20 LightGBM Feature Importances', fontsize=14)
plt.tight_layout()
plt.show()

## 🧠 Section 12: Model 5 — PyTorch Deep Neural Network (GPU Accelerated)

In [ ]:
# ============================================================
# SECTION 12 — PYTORCH DEEP TABULAR NEURAL NETWORK
# ============================================================

class TabularNN(nn.Module):
    """
    Deep residual tabular neural network for NCAA win probability.
    Architecture: BN → Linear → SiLU → Dropout × 3 layers with skip connections.
    """
    def __init__(self, input_dim, hidden_dims=(512, 256, 128, 64), dropout=0.3):
        super().__init__()
        layers = []
        prev_dim = input_dim
        self.blocks = nn.ModuleList()
        self.skips   = nn.ModuleList()

        for h in hidden_dims:
            block = nn.Sequential(
                nn.BatchNorm1d(prev_dim),
                nn.Linear(prev_dim, h),
                nn.SiLU(),
                nn.Dropout(dropout),
                nn.Linear(h, h),
                nn.SiLU(),
                nn.Dropout(dropout / 2),
            )
            skip = nn.Linear(prev_dim, h) if prev_dim != h else nn.Identity()
            self.blocks.append(block)
            self.skips.append(skip)
            prev_dim = h

        self.head = nn.Sequential(
            nn.BatchNorm1d(prev_dim),
            nn.Linear(prev_dim, 1),
            nn.Sigmoid()
        )

    def forward(self, x):
        for block, skip in zip(self.blocks, self.skips):
            x = block(x) + skip(x)
        return self.head(x).squeeze(1)


def train_nn_fold(X_tr, y_tr, X_vl, y_vl, X_te,
                   epochs=200, batch_size=512, lr=1e-3, dropout=0.3):
    """Train NN on one fold, return OOF and test predictions."""
    scaler = StandardScaler()
    X_tr_s = scaler.fit_transform(X_tr)
    X_vl_s = scaler.transform(X_vl)
    X_te_s = scaler.transform(X_te)

    # PyTorch datasets
    ds_tr = TensorDataset(
        torch.tensor(X_tr_s, dtype=torch.float32),
        torch.tensor(y_tr, dtype=torch.float32)
    )
    dl_tr = DataLoader(ds_tr, batch_size=batch_size, shuffle=True,
                       num_workers=0, pin_memory=GPU_AVAILABLE)

    X_vl_t = torch.tensor(X_vl_s, dtype=torch.float32).to(DEVICE)
    X_te_t  = torch.tensor(X_te_s, dtype=torch.float32).to(DEVICE)

    model = TabularNN(X_tr.shape[1], hidden_dims=(512, 256, 128, 64),
                      dropout=dropout).to(DEVICE)
    criterion = nn.BCELoss()
    optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    scheduler = optim.lr_scheduler.OneCycleLR(
        optimizer, max_lr=lr * 10,
        steps_per_epoch=len(dl_tr), epochs=epochs
    )

    best_val_loss = np.inf
    best_state    = None
    patience      = 20
    wait          = 0

    for epoch in range(epochs):
        model.train()
        for xb, yb in dl_tr:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            optimizer.zero_grad()
            loss = criterion(model(xb), yb)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            scheduler.step()

        # Validation
        model.eval()
        with torch.no_grad():
            vl_pred = model(X_vl_t).cpu().numpy()
        val_loss = log_loss(y_vl, vl_pred)

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_state = {k: v.clone() for k, v in model.state_dict().items()}
            wait = 0
        else:
            wait += 1
            if wait >= patience:
                break

    # Restore best model
    model.load_state_dict(best_state)
    model.eval()
    with torch.no_grad():
        oof_p  = model(X_vl_t).cpu().numpy()
        test_p = model(X_te_t).cpu().numpy()

    return oof_p, test_p, best_val_loss

# ── 5-Fold OOF for NN ─────────────────────────────────────────
oof_nn   = np.zeros(len(X_train_raw))
test_nn  = np.zeros(len(X_test_raw))

print(f"⚙️  Training PyTorch NN OOF on {DEVICE}...")
for fold, (tr_idx, vl_idx) in enumerate(kf.split(X_train_raw, y_train)):
    Xtr, Xvl = X_train_raw[tr_idx], X_train_raw[vl_idx]
    ytr, yvl = y_train[tr_idx], y_train[vl_idx]
    oof_p, test_p, best_loss = train_nn_fold(Xtr, ytr, Xvl, yvl, X_test_raw)
    oof_nn[vl_idx] = oof_p
    test_nn += test_p / N_FOLDS
    print(f"  Fold {fold+1} val log-loss: {log_loss(yvl, oof_nn[vl_idx]):.5f}  (best: {best_loss:.5f})")

oof_preds['NN']  = oof_nn
test_preds['NN'] = test_nn
print(f"\n✅ Neural Network OOF log-loss: {log_loss(y_train, oof_nn):.5f}")

## 🔀 Section 13: Ensemble & Stacking All Models

In [ ]:
# ============================================================
# SECTION 13 — ENSEMBLE & META-LEARNER STACKING
# ============================================================

# ── Individual model scores summary ──────────────────────────
model_names = list(oof_preds.keys())
print("=" * 50)
print(f"{'Model':<12}  {'OOF LogLoss':>12}")
print("=" * 50)
for mn in model_names:
    ll = log_loss(y_train, oof_preds[mn])
    print(f"{mn:<12}  {ll:>12.5f}")
print("=" * 50)

# ── Method 1: Stacked OOF as meta-features ───────────────────
oof_stack   = np.column_stack([oof_preds[m] for m in model_names])
test_stack  = np.column_stack([test_preds[m] for m in model_names])

# Logistic Regression meta-learner (no regularization penalty gives pure blending)
meta_model = LogisticRegression(C=1.0, max_iter=1000, random_state=SEED)
meta_oof   = np.zeros(len(y_train))
meta_test  = np.zeros(len(X_test_raw))

meta_kf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED + 1)
for tr_idx, vl_idx in meta_kf.split(oof_stack, y_train):
    meta_model.fit(oof_stack[tr_idx], y_train[tr_idx])
    meta_oof[vl_idx] = meta_model.predict_proba(oof_stack[vl_idx])[:,1]

meta_model.fit(oof_stack, y_train)
meta_test = meta_model.predict_proba(test_stack)[:,1]

print(f"\n✅ Stacking (Logistic meta) OOF log-loss: {log_loss(y_train, meta_oof):.5f}")
print("   Meta model weights:", dict(zip(model_names, meta_model.coef_[0])))

# ── Method 2: Optimized weighted average ─────────────────────
def neg_logloss_weights(w, oof_matrix, y_true):
    w = np.array(w)
    w = np.abs(w) / np.abs(w).sum()   # normalize to sum=1
    blended = oof_matrix @ w
    blended = np.clip(blended, 1e-7, 1 - 1e-7)
    return log_loss(y_true, blended)

n_models = len(model_names)
w0 = np.ones(n_models) / n_models
result = optimize.minimize(
    neg_logloss_weights,
    w0,
    args=(oof_stack, y_train),
    method='Nelder-Mead',
    options={'maxiter': 10000, 'xatol': 1e-8, 'fatol': 1e-8}
)
opt_weights = np.abs(result.x) / np.abs(result.x).sum()
print(f"\n✅ Optimized weights: {dict(zip(model_names, np.round(opt_weights, 4)))}")

blended_oof  = oof_stack  @ opt_weights
blended_test = test_stack @ opt_weights
print(f"✅ Weighted blend OOF log-loss: {log_loss(y_train, blended_oof):.5f}")

# ── Method 3: Rank averaging ──────────────────────────────────
from scipy.stats import rankdata
rank_oof  = np.column_stack([rankdata(oof_preds[m]) for m in model_names]).mean(axis=1)
rank_test = np.column_stack([rankdata(test_preds[m]) for m in model_names]).mean(axis=1)
# Convert ranks back to probabilities via sigmoid
rank_oof_prob  = (rank_oof  - rank_oof.min())  / (rank_oof.max()  - rank_oof.min() + 1e-9)
rank_test_prob = (rank_test - rank_test.min()) / (rank_test.max() - rank_test.min() + 1e-9)
print(f"✅ Rank average OOF log-loss: {log_loss(y_train, rank_oof_prob):.5f}")

# ── Pick the best ensemble ─────────────────────────────────────
scores_ens = {
    'Stacking'       : log_loss(y_train, meta_oof),
    'Weighted Blend' : log_loss(y_train, blended_oof),
    'Rank Average'   : log_loss(y_train, rank_oof_prob),
}
best_ens_name = min(scores_ens, key=scores_ens.get)
print(f"\n🏆 Best ensemble strategy: {best_ens_name} (OOF = {scores_ens[best_ens_name]:.5f})")

if best_ens_name == 'Stacking':
    final_test_probs = meta_test
elif best_ens_name == 'Weighted Blend':
    final_test_probs = blended_test
else:
    final_test_probs = rank_test_prob

print("\nAll ensemble strategies:")
for k, v in sorted(scores_ens.items(), key=lambda x: x[1]):
    print(f"  {k:<20}: {v:.5f}")

## 🎯 Section 14: Calibration & Prediction Clipping

In [ ]:
# ============================================================
# SECTION 14 — CALIBRATION & CLIPPING
# ============================================================
from sklearn.isotonic import IsotonicRegression
from sklearn.linear_model import LogisticRegression as LR

# ── Isotonic Regression Calibration ──────────────────────────
# Fit on OOF predictions (in-fold = unbiased estimates)
def isotonic_calibrate(oof_probs, y_true, test_probs):
    """Isotonic regression calibration (non-parametric, monotone)."""
    ir = IsotonicRegression(out_of_bounds='clip')
    ir.fit(oof_probs, y_true)
    calibrated_test = ir.predict(test_probs)
    calibrated_oof  = ir.predict(oof_probs)
    return calibrated_oof, calibrated_test

# Determine best OOF predictions based on best ensemble
if best_ens_name == 'Stacking':
    best_oof_probs = meta_oof
elif best_ens_name == 'Weighted Blend':
    best_oof_probs = blended_oof
else:
    best_oof_probs = rank_oof_prob

cal_oof, cal_test = isotonic_calibrate(best_oof_probs, y_train, final_test_probs)
print(f"Before calibration OOF log-loss: {log_loss(y_train, best_oof_probs):.5f}")
print(f"After  calibration OOF log-loss: {log_loss(y_train, cal_oof):.5f}")

# ── Platt Scaling (Sigmoid Calibration) as comparison ────────
platt = LR(C=1.0, max_iter=1000, random_state=SEED)
platt.fit(best_oof_probs.reshape(-1, 1), y_train)
platt_oof  = platt.predict_proba(best_oof_probs.reshape(-1, 1))[:,1]
platt_test = platt.predict_proba(final_test_probs.reshape(-1, 1))[:,1]
print(f"Platt scaling OOF log-loss     : {log_loss(y_train, platt_oof):.5f}")

# ── Pick best calibration ─────────────────────────────────────
if log_loss(y_train, cal_oof) <= log_loss(y_train, platt_oof):
    final_probs_calibrated = cal_test
    print("\n✅ Using: Isotonic calibration")
else:
    final_probs_calibrated = platt_test
    print("\n✅ Using: Platt scaling calibration")

# ── Clipping — key competition tip ───────────────────────────
# Clipping to [0.025, 0.975] avoids extreme log-loss on edge predictions
# Per top Kaggle notebooks and forum discussions
CLIP_LOW, CLIP_HIGH = 0.025, 0.975
final_probs_clipped = np.clip(final_probs_calibrated, CLIP_LOW, CLIP_HIGH)
print(f"\n✅ Predictions clipped to [{CLIP_LOW}, {CLIP_HIGH}]")
print(f"   Min : {final_probs_clipped.min():.4f}")
print(f"   Max : {final_probs_clipped.max():.4f}")
print(f"   Mean: {final_probs_clipped.mean():.4f}")

# ── Calibration Curve ────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Reliability diagram — before calibration
fraction_of_pos, mean_pred = calibration_curve(y_train, best_oof_probs, n_bins=20)
axes[0].plot(mean_pred, fraction_of_pos, 's-', label='Before Calibration', color='royalblue')
axes[0].plot([0,1],[0,1],'k--', label='Perfect Calibration')
axes[0].set_xlabel('Mean Predicted Probability')
axes[0].set_ylabel('Fraction of Positives')
axes[0].set_title('Calibration Curve (Before)')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# After calibration
fraction_of_pos2, mean_pred2 = calibration_curve(y_train, cal_oof, n_bins=20)
axes[1].plot(mean_pred2, fraction_of_pos2, 's-', label='After Calibration', color='darkorange')
axes[1].plot([0,1],[0,1],'k--', label='Perfect Calibration')
axes[1].set_xlabel('Mean Predicted Probability')
axes[1].set_ylabel('Fraction of Positives')
axes[1].set_title('Calibration Curve (After Isotonic)')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.suptitle('Reliability Diagrams — NCAA Tournament Predictions', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

# ── Distribution of final predictions ─────────────────────────
plt.figure(figsize=(10, 4))
plt.hist(final_probs_clipped, bins=100, color='steelblue', edgecolor='white', alpha=0.8)
plt.axvline(0.5, color='red', linestyle='--', label='0.5')
plt.xlabel('Predicted Win Probability')
plt.ylabel('Count')
plt.title('Distribution of Final Predictions')
plt.legend()
plt.tight_layout()
plt.show()

## 📤 Section 15: Generate Final Submission File

In [ ]:
# ============================================================
# SECTION 15 — FINAL SUBMISSION
# ============================================================

# ── Map predictions back to submission IDs ───────────────────
TestFull_sorted = TestFull.reset_index(drop=True)
TestFull_sorted['FinalPred'] = final_probs_clipped

# Build ID string → prediction mapping
pred_map = dict(zip(
    TestFull_sorted['Season'].astype(str) + '_' +
    TestFull_sorted['Team1'].astype(str)  + '_' +
    TestFull_sorted['Team2'].astype(str),
    TestFull_sorted['FinalPred']
))

# Map onto the original submission (preserves row order & all IDs)
sub_final = sub_raw[['ID']].copy()
sub_final['Pred'] = sub_final['ID'].map(pred_map)

# Fallback for any unmapped IDs (shouldn't happen, but safety net)
null_count = sub_final['Pred'].isna().sum()
if null_count > 0:
    print(f"⚠️  {null_count} IDs not mapped — filling with 0.5")
    sub_final['Pred'] = sub_final['Pred'].fillna(0.5)

# Final clip
sub_final['Pred'] = sub_final['Pred'].clip(CLIP_LOW, CLIP_HIGH)

# ── Save submission ───────────────────────────────────────────
SUB_PATH = os.path.join(OUTPUT_PATH, 'submission.csv')
sub_final[['ID','Pred']].to_csv(SUB_PATH, index=False)
print(f"✅ Submission saved  → {SUB_PATH}")
print(f"   Total rows        : {len(sub_final):,}")
print(f"   Null predictions  : {sub_final['Pred'].isna().sum()}")

# ── Final Summary ─────────────────────────────────────────────
print("\n" + "=" * 60)
print("📊 FINAL RESULTS SUMMARY")
print("=" * 60)
print(f"\n{'Training samples':<25}: {len(y_train):,}")
print(f"{'Test predictions':<25}: {len(sub_final):,}")
print(f"{'Feature count':<25}: {len(FEAT_COLS)}")

print(f"\n{'Model':<20} {'OOF LogLoss':>12}")
print("-" * 34)
for mn in model_names:
    ll = log_loss(y_train, oof_preds[mn])
    print(f"{mn:<20} {ll:>12.5f}")
print("-" * 34)
print(f"{'Best Ensemble':<20} {log_loss(y_train, best_oof_probs):>12.5f}  ({best_ens_name})")
print(f"{'After Calibration':<20} {log_loss(y_train, cal_oof):>12.5f}")

print(f"\n{'Pred Min':<20}: {sub_final['Pred'].min():.4f}")
print(f"{'Pred Max':<20}: {sub_final['Pred'].max():.4f}")
print(f"{'Pred Mean':<20}: {sub_final['Pred'].mean():.4f}")
print(f"{'Pred Std':<20}: {sub_final['Pred'].std():.4f}")
print("=" * 60)

print("\nFirst 10 rows of submission:")
display(sub_final.head(10))

## 🔍 Section 16: Bonus — EDA, Feature Correlations & Prediction Insights

In [ ]:
# ============================================================
# SECTION 16 — EDA & INSIGHTS
# ============================================================

# ── 1. Historical upset rate by seed matchup (Men's) ─────────
print("⚙️  Computing upset rates by seed matchup...")
tourney_with_seeds = MTourneyCompact.merge(
    MTourneySeeds[['Season','TeamID','SeedNum']].rename(columns={'TeamID':'WTeamID','SeedNum':'WSeed'}),
    on=['Season','WTeamID'], how='left'
).merge(
    MTourneySeeds[['Season','TeamID','SeedNum']].rename(columns={'TeamID':'LTeamID','SeedNum':'LSeed'}),
    on=['Season','LTeamID'], how='left'
)
tourney_with_seeds['LowerSeedWins'] = (tourney_with_seeds['WSeed'] > tourney_with_seeds['LSeed']).astype(int)

seed_matchup = tourney_with_seeds.groupby(['WSeed','LSeed']).agg(
    Games  = ('LowerSeedWins','count'),
    Upsets = ('LowerSeedWins','sum'),
).reset_index()
seed_matchup['UpsetRate'] = seed_matchup['Upsets'] / seed_matchup['Games']

# Classic seed matchups: 1v16, 2v15, 5v12, 8v9
classic = [(1,16),(2,15),(3,14),(4,13),(5,12),(6,11),(7,10),(8,9)]
print("\nUpset rates for classic seed matchups:")
print(f"{'Matchup':<12} {'Games':>6} {'Upsets':>7} {'Upset%':>8}")
print("-" * 36)
for s1, s2 in classic:
    r = seed_matchup[(seed_matchup['LSeed']==s1) & (seed_matchup['WSeed']==s2)]
    if len(r) > 0:
        row = r.iloc[0]
        print(f"  {s1} vs {s2:<6}  {int(row['Games']):>5}  {int(row['Upsets']):>6}  {row['UpsetRate']*100:>7.1f}%")

# ── 2. Elo distribution for tournament teams ─────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

elo_m_2025 = EloDF[(EloDF['Season']==2025) & (EloDF['Gender']=='M')]
axes[0].hist(elo_m_2025['Elo'], bins=40, color='royalblue', edgecolor='white')
axes[0].axvline(elo_m_2025['Elo'].mean(), color='red', linestyle='--',
                label=f"Mean: {elo_m_2025['Elo'].mean():.0f}")
axes[0].set_title("Elo Distribution — Men's Teams 2025")
axes[0].set_xlabel('Elo Rating')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Win probability vs Elo differential
elo_diff = np.linspace(-600, 600, 300)
win_prob_elo = 1 / (1 + 10**(-elo_diff/400))
axes[1].plot(elo_diff, win_prob_elo, color='darkorange', lw=2)
axes[1].axhline(0.5, color='gray', linestyle='--', alpha=0.5)
axes[1].axvline(0, color='gray', linestyle='--', alpha=0.5)
axes[1].fill_between(elo_diff, 0.5, win_prob_elo, where=win_prob_elo > 0.5,
                      alpha=0.2, color='green', label='Favored')
axes[1].fill_between(elo_diff, win_prob_elo, 0.5, where=win_prob_elo < 0.5,
                      alpha=0.2, color='red', label='Underdog')
axes[1].set_xlabel('Elo Differential (Team1 - Team2)')
axes[1].set_ylabel('Win Probability for Team1')
axes[1].set_title('Elo Win Probability Curve')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# ── 3. Feature correlation with target ───────────────────────
corr_feats = pd.DataFrame(X_train_raw, columns=FEAT_COLS)
corr_feats['Target'] = y_train
top_corr = corr_feats.corr()['Target'].abs().nlargest(25).drop('Target')

plt.figure(figsize=(10, 7))
top_corr.sort_values().plot(kind='barh', color='mediumpurple')
plt.title('Top 25 Features Correlated with Win (|r|)', fontsize=14)
plt.xlabel('|Pearson Correlation|')
plt.tight_layout()
plt.show()
print("✅ Top 5 features by correlation with target:")
print(top_corr.head())

# ── 4. Model prediction scatter ──────────────────────────────
fig, ax = plt.subplots(figsize=(8, 6))
for mn in model_names:
    ax.scatter(oof_preds[mn][:500], y_train[:500] + np.random.normal(0, 0.01, 500),
               alpha=0.2, s=10, label=mn)
ax.set_xlabel('Predicted Probability'); ax.set_ylabel('Actual (jittered)')
ax.set_title('Model Prediction Scatter (first 500 samples)')
ax.legend()
plt.tight_layout()
plt.show()

## 🚀 Section 17: Bonus — Elo-Only Baseline Submission (Fast, Interpretable)

In [ ]:
# ============================================================
# SECTION 17 — ELO-ONLY BASELINE SUBMISSION (sanity check)
# ============================================================

def elo_win_prob(elo1, elo2):
    """Standard Elo expected win probability."""
    return 1.0 / (1.0 + 10.0 ** ((elo2 - elo1) / 400.0))

# Parse IDs from submission
elo_sub = sub_raw[['ID']].copy()
split_ids = elo_sub['ID'].str.split('_', expand=True)
elo_sub['Season'] = split_ids[0].astype(int)
elo_sub['Team1']  = split_ids[1].astype(int)
elo_sub['Team2']  = split_ids[2].astype(int)
elo_sub['Gender'] = elo_sub['Team1'].apply(lambda x: 'W' if x >= 3000 else 'M')

# Merge Elo for both teams
elo_sub = elo_sub.merge(
    EloDF.rename(columns={'TeamID':'Team1','Elo':'Elo1'}),
    on=['Season','Team1','Gender'], how='left'
).merge(
    EloDF.rename(columns={'TeamID':'Team2','Elo':'Elo2'}),
    on=['Season','Team2','Gender'], how='left'
)

# Fill missing Elo with global median
elo_med = EloDF['Elo'].median()
elo_sub['Elo1'] = elo_sub['Elo1'].fillna(elo_med)
elo_sub['Elo2'] = elo_sub['Elo2'].fillna(elo_med)

elo_sub['Pred'] = elo_sub.apply(lambda r: elo_win_prob(r['Elo1'], r['Elo2']), axis=1)
elo_sub['Pred'] = elo_sub['Pred'].clip(CLIP_LOW, CLIP_HIGH)

ELO_SUB_PATH = os.path.join(OUTPUT_PATH, 'submission_elo_baseline.csv')
elo_sub[['ID','Pred']].to_csv(ELO_SUB_PATH, index=False)
print(f"✅ Elo baseline saved → {ELO_SUB_PATH}")
print(f"   Rows      : {len(elo_sub):,}")
print(f"   Pred mean : {elo_sub['Pred'].mean():.4f}")
print(f"\nSample:\n{elo_sub[['ID','Pred']].head(10).to_string(index=False)}")

# ── Validate on historical tourney data ──────────────────────
tourney_val = AllTourneyCompact[AllTourneyCompact['Season'] >= 2003].copy()
tourney_val = tourney_val.merge(
    EloDF.rename(columns={'TeamID':'WTeamID','Elo':'WElo'}),
    on=['Season','WTeamID','Gender'], how='left'
).merge(
    EloDF.rename(columns={'TeamID':'LTeamID','Elo':'LElo'}),
    on=['Season','LTeamID','Gender'], how='left'
)
tourney_val = tourney_val.dropna(subset=['WElo','LElo'])
tourney_val['WProb'] = tourney_val.apply(
    lambda r: elo_win_prob(r['WElo'], r['LElo']), axis=1
).clip(0.025, 0.975)
elo_baseline_ll = log_loss(np.ones(len(tourney_val)), tourney_val['WProb'])
print(f"\n📊 Elo-only historical tourney log-loss: {elo_baseline_ll:.5f}")
print(f"   (Based on {len(tourney_val):,} games, 2003–2025)")

## 📦 Section 18: Install Required Packages (Run First on Kaggle)

In [ ]:
# ============================================================
# SECTION 18 — PACKAGE INSTALLATION CHECK
# Run this cell FIRST if any imports fail
# ============================================================

import subprocess, sys

def install_package(pkg):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg, '-q'])

required_packages = {
    'xgboost'   : 'xgboost>=2.0',
    'lightgbm'  : 'lightgbm>=4.0',
    'catboost'  : 'catboost>=1.2',
    'optuna'    : 'optuna>=3.0',
    'torch'     : 'torch',
    'tqdm'      : 'tqdm',
}

print("Checking required packages...")
for pkg_name, pkg_install in required_packages.items():
    try:
        __import__(pkg_name)
        print(f"  ✅ {pkg_name} already installed")
    except ImportError:
        print(f"  ⚙️  Installing {pkg_install}...")
        install_package(pkg_install)
        print(f"  ✅ {pkg_name} installed")

print("\n✅ All required packages available!")
print("\n💡 On Kaggle: Enable GPU in Settings → Accelerator → GPU T4 x2 or P100")
print("💡 XGBoost GPU: tree_method='gpu_hist' or device='cuda'")
print("💡 LightGBM GPU: device='gpu'")
print("💡 CatBoost GPU: task_type='GPU'")
print("💡 PyTorch GPU: model.to(torch.device('cuda'))")